# L15a: Initial Value Problems and Explicit Methods
In this lecture, we will explore initial value problems and explicit numerical methods for solving them. Before we dive into the details, let's set up the context for our discussion, what is an initial value problem?

> __The initial value problem__
>
> Let $t_0$ be the starting time and $t_f>t_0$ the final time. The state vector $\mathbf{x}(t)\in\mathbb{R}^{n}$ holds the $n$ state variables of the model at time $t\in[t_0,t_f]$. The input vector $\mathbf{u}(t)\in\mathbb{R}^{d}$ holds the $d$ inputs, given as a function $\mathbf{u}:[t_0,t_f]\rightarrow\mathbb{R}^{d}$ that is continuous except at a finite number of jumps.
>
> In our gene expression model, $\mathbf{x}$ is the concentration vector of a circuit with $g$ genes. It holds the $g$ messenger RNA (mRNA) concentrations followed by the $g$ protein concentrations, which gives $n=2g$ species. The input vector $\mathbf{u}\in\mathbb{R}^{g}$ holds the transcriptional control $u_j\in[0,1]$ of each gene $j$, which scales the transcription rate of gene $j$.
>
> The right-hand side $\mathbf{f}:\mathbb{R}^{n}\times\mathbb{R}^{d}\rightarrow\mathbb{R}^{n}$ gives the rate of change of every state as a function of the current state and input (these are our material balance equations). An __initial value problem__ asks for a function $\mathbf{x}:[t_0,t_f]\rightarrow\mathbb{R}^{n}$ whose time derivative $\dot{\mathbf{x}}=\mathrm{d}\mathbf{x}/\mathrm{d}t$ satisfies the differential equation, and which starts at the initial state $\mathbf{x}_0\in\mathbb{R}^{n}$:
>
> $$
> \underbrace{\dot{\mathbf{x}}(t)=\mathbf{f}(\mathbf{x}(t),\mathbf{u}(t))}_{\text{differential equation}},
> \qquad
> \underbrace{\mathbf{x}(t_0)=\mathbf{x}_0}_{\text{initial condition}}.
> $$
>
> We assume that $\mathbf{f}$ is continuous and that it is __Lipschitz continuous__ in the state. This means that the change in the rates is at most a fixed multiple $L>0$ of the change in the state:
>
> $$
> \|\mathbf{f}(\mathbf{x},\mathbf{u})-\mathbf{f}(\mathbf{y},\mathbf{u})\|\le L\,\|\mathbf{x}-\mathbf{y}\|\quad\text{for all }\mathbf{x},\mathbf{y}\in\mathbb{R}^{n}\text{ and every input value }\mathbf{u},
> $$
>
> where $\|\cdot\|$ is the Euclidean length of a vector. For our gene expression model, we need this bound only for nonnegative concentrations. Under this assumption, the problem has exactly one solution. At a jump of the input, the solution stays continuous, but its derivative can jump. The differential equation holds at all other times.
>
> Today, we will focus on __explicit methods__ for solving initial value problems. These methods compute the next state based on the current state and an estimate of the slope of the solution.

To compute an approximate solution, we choose a number of steps $N\ge1$ and the step size $\Delta t=(t_f-t_0)/N$. The grid points are $t_k=t_0+k\Delta t$ for $k=0,1,\dots,N$, where $k$ counts the completed steps. We write $\mathbf{x}_k\in\mathbb{R}^{n}$ for our approximation to $\mathbf{x}(t_k)$, starting from the initial state $\mathbf{x}_0$.

A __one-step method__ computes the next state from the current state and an estimate $\mathbf{s}_k\in\mathbb{R}^{n}$ of the average slope over the step. For a given right-hand side and input, the estimate depends only on $t_k$, $\mathbf{x}_k$, and $\Delta t$:

$$
\mathbf{x}_{k+1}=\mathbf{x}_k+\Delta t\,\mathbf{s}_k,\qquad k=0,1,\dots,N-1.
$$

We will derive simple methods we can write by hand to see how the slope estimate controls accuracy and stability. In practice we use a solver library such as [DifferentialEquations.jl](https://docs.sciml.ai/DiffEqDocs/stable/). Our course environment loads its ordinary differential equation solvers through [OrdinaryDiffEq.jl](https://docs.sciml.ai/OrdinaryDiffEq/stable/).

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Derive the explicit Euler update:__ Integrate the differential equation over a step and approximate the integral using the slope at the start. Explain how the error from one step differs from the error over a whole trajectory.
> * __Determine when Euler preserves decay:__ Compute the factor by which Euler multiplies the solution of a linear test equation at each step. Use its magnitude to find the allowed step sizes for real and complex eigenvalues.
> * __Build a Runge–Kutta step:__ Evaluate the right-hand side at trial states within a step. Relate the slope samples to the order and cost of the method.

Let's get started!

___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Compare explicit methods on gene circuits](CHEME-5800-L15a-Example-ExplicitMethods-Fall-2026.ipynb). In this example, we will compare Euler and fourth-order Runge–Kutta with a library solution through a pulse of an inducer, a chemical added to the medium that changes the transcriptional controls. We will measure how the error changes as we halve the step and test whether Euler preserves the decay of an oscillating circuit.

___

## Explicit Euler
One of the simplest methods for solving an initial value problem is the __explicit Euler method__. It uses the slope at the start of a step to estimate the change in state over that step. Let's start from the differential equation and integrate both sides between adjacent grid points, with $s\in[t_k,t_{k+1}]$ the time inside the step:

$$
\begin{aligned}
\dot{\mathbf{x}}(t) &= \mathbf{f}(\mathbf{x}(t),\mathbf{u}(t)) &&\quad\text{(the differential equation)}\\
\int_{t_k}^{t_{k+1}}\dot{\mathbf{x}}(s)\,\mathrm{d}s &= \int_{t_k}^{t_{k+1}}\mathbf{f}(\mathbf{x}(s),\mathbf{u}(s))\,\mathrm{d}s &&\quad\text{(integrate both sides from }t_k\text{ to }t_{k+1}\text{)}\\
\mathbf{x}(t_{k+1})-\mathbf{x}(t_k) &= \int_{t_k}^{t_{k+1}}\mathbf{f}(\mathbf{x}(s),\mathbf{u}(s))\,\mathrm{d}s &&\quad\text{(fundamental theorem of calculus)}\\
\mathbf{x}(t_{k+1}) &= \mathbf{x}(t_k)+\int_{t_k}^{t_{k+1}}\mathbf{f}(\mathbf{x}(s),\mathbf{u}(s))\,\mathrm{d}s &&\quad\text{(move }\mathbf{x}(t_k)\text{ to the right)}
\end{aligned}
$$

The different methods that we explore differ in how they approximate the integral on the right-hand side. The simplest is Euler's method, which uses the slope at the start of the step.

> __The explicit Euler step__ 
>
> Approximate the integrand by its value at the left endpoint. The resulting rectangle has width $\Delta t$ and height given by the slope at the start:
>
> $$
> \begin{aligned}
> \mathbf{x}(t_{k+1}) &\approx \mathbf{x}(t_k)+\Delta t\,\underbrace{\mathbf{f}(\mathbf{x}(t_k),\mathbf{u}(t_k))}_{\text{slope at the start}} &&\quad\text{(hold the integrand at its value at }t_k\text{)}\\
> \mathbf{x}_{k+1} &= \mathbf{x}_k+\Delta t\,\mathbf{f}(\mathbf{x}_k,\mathbf{u}(t_k)) &&\quad\text{(replace each }\mathbf{x}(t_k)\text{ by its approximation }\mathbf{x}_k\text{)}
> \end{aligned}
> $$
>
> Starting from $\mathbf{x}_0$, the update gives $\mathbf{x}_1,\dots,\mathbf{x}_N$ in turn. The method is __explicit__ because we can evaluate the right-hand side and compute the next state directly.

We measure the accuracy of a method with two errors. The __local error__ $\boldsymbol{\ell}_{k+1}\in\mathbb{R}^{n}$ is the error of one step that starts from the exact state $\mathbf{x}(t_k)$. The __global error__ $\mathbf{e}_k=\mathbf{x}_k-\mathbf{x}(t_k)\in\mathbb{R}^{n}$ is the difference between the computed and exact states after $k$ steps.

Suppose that $\mathbf{f}$ has continuous first derivatives and that the input has a continuous derivative within each step, for example a pulse whose edges fall on grid points. At a jump on a grid point $t_k$, the value $\mathbf{u}(t_k)$ is the input just after the jump. The second derivative $\ddot{\mathbf{x}}(t)\in\mathbb{R}^{n}$ is then continuous within each step. Let $M$ be a finite upper bound on $\|\ddot{\mathbf{x}}(t)\|$ within the steps. We bound the local error of Euler using the exact update:

$$
\begin{aligned}
\boldsymbol{\ell}_{k+1} &= \mathbf{x}(t_{k+1})-\mathbf{x}(t_k)-\Delta t\,\mathbf{f}(\mathbf{x}(t_k),\mathbf{u}(t_k)) &&\quad\text{(the exact state minus one Euler step from }\mathbf{x}(t_k)\text{)}\\
&= \int_{t_k}^{t_{k+1}}\left(\dot{\mathbf{x}}(s)-\dot{\mathbf{x}}(t_k)\right)\mathrm{d}s &&\quad\text{(exact update, and }\Delta t\,\dot{\mathbf{x}}(t_k)\text{ written as an integral)}\\
&= \int_{t_k}^{t_{k+1}}\int_{t_k}^{s}\ddot{\mathbf{x}}(r)\,\mathrm{d}r\,\mathrm{d}s &&\quad\text{(fundamental theorem of calculus for }\dot{\mathbf{x}}\text{, with }r\in[t_k,s]\text{)}\\
\|\boldsymbol{\ell}_{k+1}\| &\le \int_{t_k}^{t_{k+1}}M\,(s-t_k)\,\mathrm{d}s=\frac{M}{2}\,\Delta t^{2} &&\quad\text{(the inner integral has length at most }M(s-t_k)\text{)}
\end{aligned}
$$

The global error starts at $\mathbf{e}_0=\mathbf{0}$. Each step carries the previous error forward and introduces a new local error. We subtract the definition of the local error from the Euler step and bound each term:

$$
\begin{aligned}
\mathbf{e}_{k+1} &= \mathbf{e}_k+\Delta t\left(\mathbf{f}(\mathbf{x}_k,\mathbf{u}(t_k))-\mathbf{f}(\mathbf{x}(t_k),\mathbf{u}(t_k))\right)-\boldsymbol{\ell}_{k+1} &&\quad\text{(Euler step minus the definition of }\boldsymbol{\ell}_{k+1}\text{)}\\
\|\mathbf{e}_{k+1}\| &\le (1+L\Delta t)\,\|\mathbf{e}_k\|+\frac{M}{2}\,\Delta t^{2} &&\quad\text{(triangle inequality, Lipschitz bound, and local error bound)}\\
\|\mathbf{e}_k\| &\le \frac{M}{2}\,\Delta t^{2}\sum_{j=0}^{k-1}(1+L\Delta t)^{j}=\frac{M\Delta t}{2L}\left((1+L\Delta t)^{k}-1\right) &&\quad\text{(apply the bound }k\text{ times from }\mathbf{e}_0=\mathbf{0}\text{ and sum the geometric series)}\\
&\le \frac{M\Delta t}{2L}\left(e^{L(t_k-t_0)}-1\right) &&\quad\text{(}1+L\Delta t\le e^{L\Delta t}\text{ and }k\Delta t=t_k-t_0\text{)}
\end{aligned}
$$

The bound is a constant times $\Delta t$. The constant depends on $M$, $L$, and the length of the interval, but not on the step size. It grows exponentially with the length of the interval.

We write $O(\Delta t^p)$ for a quantity whose size is at most $C\Delta t^p$ for every small enough step $\Delta t>0$, where the constant $C$ does not depend on $\Delta t$ or $k$. The __order__ of a method is the largest exponent $p$ for which $\max_{0\le k\le N}\|\mathbf{e}_k\|=O(\Delta t^p)$ on every problem with a smooth enough right-hand side and input.

Euler is first order, with local error $O(\Delta t^2)$ and global error $O(\Delta t)$. When the term proportional to $\Delta t$ dominates the global error, halving the step approximately halves that error.

### Stability
A __steady state__ is a state $\mathbf{x}^{\star}\in\mathbb{R}^{n}$ and a constant input $\mathbf{u}^{\star}\in\mathbb{R}^{d}$ at which every rate vanishes, $\mathbf{f}(\mathbf{x}^{\star},\mathbf{u}^{\star})=\mathbf{0}$. When $\mathbf{f}$ has continuous first derivatives, the __Jacobian__ $\mathbf{A}\in\mathbb{R}^{n\times n}$ records how each rate changes with each entry of the state at the steady state. Its entry in row $i$ and column $j$ is the partial derivative $\partial f_i/\partial x_j$ of the $i$-th rate with respect to the $j$-th state variable, evaluated at $(\mathbf{x}^{\star},\mathbf{u}^{\star})$.

With the input held at $\mathbf{u}^{\star}$, a small deviation $\Delta\mathbf{x}(t)=\mathbf{x}(t)-\mathbf{x}^{\star}$ approximately follows the linear model $\tfrac{\mathrm{d}}{\mathrm{d}t}\Delta\mathbf{x}=\mathbf{A}\,\Delta\mathbf{x}$. Suppose that $\mathbf{A}$ has $n$ linearly independent eigenvectors $\mathbf{v}_1,\dots,\mathbf{v}_n\in\mathbb{C}^{n}$ with eigenvalues $\lambda_1,\dots,\lambda_n\in\mathbb{C}$.

Writing the deviation as $\Delta\mathbf{x}(t)=\sum_{i=1}^{n}y_i(t)\,\mathbf{v}_i$ separates it into __modes__, one for each eigenvector. Substituting this sum into the linear model shows that each amplitude $y_i(t)\in\mathbb{C}$ follows the scalar equation $\dot y_i=\lambda_i y_i$.

> __The amplification factor__
>
> Euler applied to the linear model gives the computed deviations $\Delta\mathbf{x}_{k+1}=(\mathbf{I}+\Delta t\,\mathbf{A})\,\Delta\mathbf{x}_k$, where $\mathbf{I}$ is the $n\times n$ identity matrix. Because $(\mathbf{I}+\Delta t\,\mathbf{A})\mathbf{v}_i=(1+\Delta t\lambda_i)\mathbf{v}_i$, each coordinate $y_k$ of the computed deviation follows Euler applied to the __test equation__ $\dot y=\lambda y$, with $y(t)\in\mathbb{C}$ and $\lambda\in\mathbb{C}$:
>
> $$
> y_{k+1}=\underbrace{(1+\Delta t\lambda)}_{\text{Euler amplification factor}}y_k.
> $$
>
> The exact solution multiplies the amplitude by $e^{\lambda\Delta t}$ over one step, a factor with magnitude below one when $\operatorname{Re}\lambda<0$. Euler preserves decay when $|1+\Delta t\lambda|<1$. The __stability region__ of Euler is the set of scaled eigenvalues $z=\Delta t\lambda$ for which this inequality holds, the disk $\{z\in\mathbb{C}:|1+z|<1\}$ of radius one centered at $z=-1$.

For real negative $\lambda$, the condition $-1<1+\Delta t\lambda<1$ gives $0<\Delta t<2/|\lambda|$. If all the eigenvalues are real and negative, the fastest-decaying mode has the largest $|\lambda|$ and sets the limit.

Because $\mathbf{A}$ is real, its complex eigenvalues come in conjugate pairs, and both members of a pair have the same value of $|1+\Delta t\lambda|$. For a complex eigenvalue with $\operatorname{Re}\lambda<0$, expand the squared magnitude:

$$
|1+\Delta t\lambda|^2
=1+2\Delta t\operatorname{Re}\lambda+\Delta t^2|\lambda|^2.
$$

Subtract one and divide the stability inequality by the positive step size. This gives a bound of $\Delta t<-2\operatorname{Re}\lambda/|\lambda|^2$. The smallest bound over the decaying modes sets the step limit.

The quadratic term raises the amplification magnitude for an oscillatory mode. For a pure oscillation, $\lambda=i\omega$ with real angular frequency $\omega\ne0$, the linear term vanishes and Euler increases the amplitude at every step. For a decaying oscillation, the negative linear term can offset that increase if the step is small enough.

___

## Runge–Kutta Methods
A __Runge–Kutta method__ estimates the average slope by combining one or more evaluations of the right-hand side during a step. Each evaluation is called a __stage__.

> __Explicit Runge–Kutta methods__
>
> An explicit Runge–Kutta method with $q\ge1$ stages computes the stage slopes $\mathbf{k}_1,\dots,\mathbf{k}_q\in\mathbb{R}^{n}$ in order. Stage $i$ evaluates the right-hand side at a trial state built from the earlier slopes. The update then takes a weighted sum of all the slopes:
>
> $$
> \begin{aligned}
> \mathbf{k}_i &= \mathbf{f}\Big(\underbrace{\mathbf{x}_k+\Delta t\sum_{j=1}^{i-1}a_{ij}\,\mathbf{k}_j}_{\text{trial state}},\;\mathbf{u}(t_k+c_i\Delta t)\Big), &&\quad i=1,\dots,q,\\
> \mathbf{x}_{k+1} &= \mathbf{x}_k+\Delta t\sum_{i=1}^{q}b_i\,\mathbf{k}_i.
> \end{aligned}
> $$
>
> The real coefficients $a_{ij}$, the real weights $b_i$, and the nodes $c_i$ define the method. The nodes set the stage times $t_k+c_i\Delta t$, which usually lie within the step. The weights sum to one, which makes the update exact when the slope is constant. The method is __explicit__ because stage $i$ uses only the slopes of the earlier stages $j<i$.

Explicit Euler is the one-stage case, with $q=1$, $c_1=0$, and $b_1=1$. Heun's method adds a second stage at the end of the step.

> __Heun's method__
>
> Let $\mathbf{k}_1=\mathbf{f}(\mathbf{x}_k,\mathbf{u}(t_k))$ be the starting slope. An Euler step gives a trial end state $\mathbf{x}_{\mathrm{trial}}=\mathbf{x}_k+\Delta t\,\mathbf{k}_1$. Evaluate the end slope $\mathbf{k}_2=\mathbf{f}(\mathbf{x}_{\mathrm{trial}},\mathbf{u}(t_{k+1}))$ and use the average:
>
> $$
> \mathbf{x}_{k+1}=\mathbf{x}_k+
> \Delta t\,\underbrace{\frac{\mathbf{k}_1+\mathbf{k}_2}{2}}_{\text{average of the endpoint slopes}}.
> $$
>
> In the general form, $q=2$, $c_1=0$, $a_{21}=c_2=1$, and $b_1=b_2=1/2$. The method is second order on a problem with a smooth enough solution. It evaluates the right-hand side twice per step.

The classical fourth-order Runge–Kutta method (RK4) also samples the midpoint. It samples that time twice, using a new state estimate for the second evaluation.

> __The classical fourth-order Runge–Kutta step__
>
> For a step from $(t_k,\mathbf{x}_k)$, compute the four slopes in order:
>
> $$
> \begin{aligned}
> \mathbf{k}_1 &= \underbrace{\mathbf{f}(\mathbf{x}_k,\mathbf{u}(t_k))}_{\text{start}},\\[4pt]
> \mathbf{k}_2 &= \underbrace{\mathbf{f}\!\left(\mathbf{x}_k+\tfrac{\Delta t}{2}\mathbf{k}_1,\,
> \mathbf{u}(t_k+\tfrac{\Delta t}{2})\right)}_{\text{midpoint using the starting slope}},\\[4pt]
> \mathbf{k}_3 &= \underbrace{\mathbf{f}\!\left(\mathbf{x}_k+\tfrac{\Delta t}{2}\mathbf{k}_2,\,
> \mathbf{u}(t_k+\tfrac{\Delta t}{2})\right)}_{\text{midpoint using the first midpoint slope}},\\[4pt]
> \mathbf{k}_4 &= \underbrace{\mathbf{f}(\mathbf{x}_k+\Delta t\,\mathbf{k}_3,\mathbf{u}(t_{k+1}))}_{\text{end using the second midpoint slope}}.
> \end{aligned}
> $$
>
> The update weights the midpoint slopes more heavily:
>
> $$
> \mathbf{x}_{k+1}=\mathbf{x}_k+\frac{\Delta t}{6}
> \left(\mathbf{k}_1+2\mathbf{k}_2+2\mathbf{k}_3+\mathbf{k}_4\right).
> $$
>
> In the general form, $q=4$, $(c_1,c_2,c_3,c_4)=(0,\tfrac12,\tfrac12,1)$, $a_{21}=a_{32}=\tfrac12$, $a_{43}=1$, and $(b_1,b_2,b_3,b_4)=(\tfrac16,\tfrac13,\tfrac13,\tfrac16)$. Every other $a_{ij}$ is zero.

On a problem with a smooth enough solution, RK4 has local error $O(\Delta t^5)$ and global error $O(\Delta t^4)$. When the term proportional to $\Delta t^4$ dominates the global error, halving the step cuts that error by about a factor of sixteen.

Each step costs four right-hand-side evaluations, which we can use to compare the work needed to reach a given accuracy.

Applying RK4 to the test equation gives $y_{k+1}=R(z)\,y_k$ with the amplification factor $R(z)=1+z+z^2/2+z^3/6+z^4/24$, where $z=\Delta t\lambda$. On the real axis, $|R(z)|<1$ for $z$ between about $-2.785$ and $0$, compared with $-2<z<0$ for Euler. On the imaginary axis, $|R(z)|<1$ for $0<|\operatorname{Im}z|<2\sqrt{2}$, where Euler increases the amplitude at every step.

Hairer, Nørsett, and Wanner derive the order conditions of Runge–Kutta methods in [*Solving Ordinary Differential Equations I*, Chapter II](https://doi.org/10.1007/978-3-540-78862-1).

___

## Summary
An initial value problem specifies the right-hand side, the input, and the initial state on an interval of time. A one-step method moves from the current state by estimating the average slope over the next interval.

> __Key Takeaways:__
>
> * __Euler holds the slope fixed:__ We derived the update by replacing the slope over the step with its value at the start. The local error is bounded by a constant times the square of the step size, and the global error on a smooth problem is first order.
> * __Stability constrains the step size:__ We used the amplification factor to determine whether a decaying mode still decays after discretization. A slowly decaying oscillation can require a smaller step than the fastest-decaying mode.
> * __Runge–Kutta samples several slopes:__ We used trial states to take slope samples inside a step. The classical fourth-order method spends more evaluations per step to reduce the error more quickly as the grid is refined.

The step size controls both how well a method approximates the trajectory and whether a decaying mode still decays.

___